# Tech Challenge Fase 2  
## Notebook 05.2 — Data Quality Silver

### Responsabilidade do notebook

Este notebook executa as regras de qualidade da camada **Silver**.

A validação Silver possui foco semântico, estrutural e relacional, verificando:

- valores nulos em chaves obrigatórias;
- duplicidade por chave de negócio;
- valores permitidos;
- percentuais fora da faixa esperada;
- existência de colunas;
- integridade dos arquivos tratados;
- separação de registros inválidos;
- geração de resultados detalhados e consolidados.

---

### Entradas

```text
config/config.json
config/silver_metadata
config/quality_metadata
silver/<dataset>/ano=YYYY/*.csv
```

### Saídas

```text
logs/data_quality/silver/details
logs/data_quality/silver/summary
logs/data_quality/rejected/silver
logs/data_quality/history/silver
```

## 1. Contexto da qualidade Silver

A Silver é responsável por transformar dados brutos em dados padronizados e confiáveis.

```text
Bronze
 ↓
Silver  ← validação semântica e relacional
 ↓
Gold
```

As regras de qualidade não substituem as transformações da Silver.  
Elas verificam se os dados tratados atendem aos critérios definidos pelo projeto.

## 2. Importação das bibliotecas

Nesta etapa importamos recursos para:

- leitura das configurações;
- leitura dos metadados Silver;
- leitura dos arquivos CSV;
- aplicação das regras;
- persistência de resultados;
- criação de schemas explícitos.

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType
)

## 3. Leitura das configurações oficiais

Os caminhos são carregados do `config.json`.

Essa abordagem mantém o notebook alinhado ao setup e aos orquestradores Bronze, Silver, Gold e Quality.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    dbutils.fs.head(CONFIG_FILE_PATH)
)

SILVER_PATH = config["paths"]["silver_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

QUALITY_ROOT_PATH = f"{LOG_PATH}/data_quality"
QUALITY_SILVER_PATH = f"{QUALITY_ROOT_PATH}/silver"
QUALITY_DETAILS_PATH = f"{QUALITY_SILVER_PATH}/details"
QUALITY_SUMMARY_PATH = f"{QUALITY_SILVER_PATH}/summary"
QUALITY_REJECTED_PATH = f"{QUALITY_ROOT_PATH}/rejected/silver"
QUALITY_HISTORY_PATH = f"{QUALITY_ROOT_PATH}/history/silver"

print("SILVER_PATH:", SILVER_PATH)
print("QUALITY_SILVER_PATH:", QUALITY_SILVER_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Criação dos diretórios de saída

Criamos os diretórios utilizados para:

- resultados detalhados;
- resumos executivos;
- registros inválidos;
- histórico de qualidade.

In [0]:
for path in [
    QUALITY_SILVER_PATH,
    QUALITY_DETAILS_PATH,
    QUALITY_SUMMARY_PATH,
    QUALITY_REJECTED_PATH,
    QUALITY_HISTORY_PATH
]:
    dbutils.fs.mkdirs(path)

print("Diretórios de qualidade Silver criados/validados.")

## 5. Leitura da `silver_metadata`

A `silver_metadata` informa os caminhos e nomes dos arquivos Silver gerados pelos notebooks de tratamento.

In [0]:
silver_metadata_path = (
    f"{CONFIG_PATH}/silver_metadata"
)

df_silver_metadata = (
    spark.read
    .parquet(silver_metadata_path)
)

display(
    df_silver_metadata
    .orderBy("dataset", "ano")
)

## 6. Leitura das regras Silver

O notebook consome apenas regras ativas da camada Silver:

```text
layer = silver
enabled = true
```

In [0]:
quality_metadata_path = (
    f"{CONFIG_PATH}/quality_metadata"
)

df_quality_metadata = (
    spark.read
    .parquet(quality_metadata_path)
)

df_silver_rules = (
    df_quality_metadata
    .filter(
        (F.col("layer") == "silver")
        & F.col("enabled")
    )
    .orderBy("dataset", "rule_id")
)

display(df_silver_rules)

silver_rules = [
    row.asDict()
    for row in df_silver_rules.collect()
]

print(
    "Quantidade de regras Silver:",
    len(silver_rules)
)

## 7. Schema dos resultados de qualidade

Cada execução de regra gera um registro contendo:

- dataset e ano;
- regra aplicada;
- status;
- quantidade de registros avaliados;
- quantidade de registros inválidos;
- percentual inválido;
- tolerância;
- mensagem;
- caminho avaliado.

In [0]:
schema_quality_result = StructType([
    StructField("layer", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("blocking", BooleanType(), False),
    StructField("tolerance_percent", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("records_evaluated", LongType(), False),
    StructField("invalid_records", LongType(), False),
    StructField("invalid_percent", DoubleType(), False),
    StructField("message", StringType(), True),
    StructField("evaluated_path", StringType(), True),
    StructField("execution_date", StringType(), False),
    StructField("evaluated_at", StringType(), False)
])

## 8. Funções auxiliares

Nesta etapa implementamos funções para:

- verificar existência do arquivo;
- ler CSV Silver;
- normalizar listas de colunas;
- calcular status;
- validar tipos de regra.

In [0]:
def file_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def split_columns(column_name: str) -> list:
    if not column_name:
        return []

    return [
        item.strip()
        for item in column_name.split(",")
        if item.strip()
    ]


def calculate_status(
    invalid_percent: float,
    tolerance_percent: float
) -> str:
    if invalid_percent <= tolerance_percent:
        return "APROVADO"

    attention_limit = max(
        tolerance_percent * 2,
        tolerance_percent + 1.0
    )

    if invalid_percent <= attention_limit:
        return "ATENCAO"

    return "REPROVADO"


def read_silver_csv(
    file_path: str
):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .csv(file_path)
    )

    # Normaliza espaços acidentais nos
    # nomes das colunas antes da
    # execução das regras.

    for coluna in df.columns:

        coluna_normalizada = (
            coluna.strip()
        )

        if coluna_normalizada != coluna:

            df = (
                df.withColumnRenamed(
                    coluna,
                    coluna_normalizada
                )
            )

    return df


def numeric_column(column_name: str):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.trim(F.col(column_name).cast("string")),
                "%",
                ""
            ),
            ",",
            "."
        )
        .cast("double")
    )

## 9. Mapeamento de colunas equivalentes

Alguns arquivos podem preservar nomes diferentes para a mesma chave.

Exemplo:

```text
CO_UF
CD_UF
```

A função abaixo permite localizar uma coluna equivalente sem alterar o arquivo Silver.

In [0]:
column_aliases = {
    "CO_UF": [
        "CO_UF",
        "CD_UF",
        "COD_UF"
    ],
    "CO_MUNICIPIO": [
        "CO_MUNICIPIO",
        "CD_MUNICIPIO"
    ],
    "ID_ALUNO": [
        "ID_ALUNO",
        "NU_ID_ALUNO"
    ],
    "ANO": [
        "ANO",
        "NU_ANO_AVALIACAO",
        "_ano_referencia"
    ],
    "PC_ALUNO_ALFABETIZADO": [
        "PC_ALUNO_ALFABETIZADO",
        "PC_ALUNO_ALFABETIZADO_2023",
        "PC_ALUNO_ALFABETIZADO_2024",
        "PC_ALUNO_ALFABETIZADO_2025"
    ]
}


def resolve_column(
    expected_column: str,
    available_columns: list
):
    if expected_column in available_columns:
        return expected_column

    aliases = column_aliases.get(
        expected_column,
        []
    )

    for alias in aliases:
        if alias in available_columns:
            return alias

    return None

## 10. Execução das regras Silver

Tipos de regra suportados:

- `not_null`;
- `unique`;
- `allowed_values`;
- `between`.

Os registros inválidos são armazenados separadamente por regra, dataset e ano.

In [0]:
quality_results = []
rejected_records_paths = []

metadata_rows = [
    row.asDict()
    for row in (
        df_silver_metadata
        .orderBy("dataset", "ano")
        .collect()
    )
]

for metadata_item in metadata_rows:
    dataset = metadata_item["dataset"]
    ano = int(metadata_item["ano"])

    dataset_rules = [
        rule
        for rule in silver_rules
        if rule["dataset"] == dataset
    ]

    if not dataset_rules:
        continue

    silver_file_path = (
        f"{metadata_item['silver_path']}/"
        f"{metadata_item['silver_file_name']}"
    )

    file_available = file_exists(
        silver_file_path
    )

    df_dataset = None
    records_evaluated = 0
    available_columns = []

    if file_available:
        try:
            df_dataset = read_silver_csv(
                silver_file_path
            )

            if (
                "ANO" not in df_dataset.columns
                and "NU_ANO_AVALIACAO"
                not in df_dataset.columns
                and "_ano_referencia"
                not in df_dataset.columns
            ):
                df_dataset = (
                    df_dataset
                    .withColumn(
                        "ANO",
                        F.lit(ano)
                    )
                )

            records_evaluated = (
                df_dataset.count()
            )

            available_columns = (
                df_dataset.columns
            )

        except Exception as e:
            file_available = False
            read_error = str(e)
    else:
        read_error = (
            "Arquivo Silver não encontrado."
        )

    for rule in dataset_rules:
        rule_type = rule["rule_type"]
        expected_columns = split_columns(
            rule["column_name"]
        )

        resolved_columns = []
        missing_columns = []

        for expected_column in expected_columns:
            resolved = resolve_column(
                expected_column,
                available_columns
            )

            if resolved:
                resolved_columns.append(
                    resolved
                )
            else:
                missing_columns.append(
                    expected_column
                )

        invalid_df = None
        invalid_records = 0
        invalid_percent = 0.0
        message = ""

        if not file_available:
            invalid_records = 1
            invalid_percent = 100.0
            message = (
                f"Arquivo indisponível: "
                f"{read_error}"
            )

        elif records_evaluated == 0:
            invalid_records = 1
            invalid_percent = 100.0
            message = (
                "Arquivo Silver vazio."
            )

        elif missing_columns:
            invalid_records = records_evaluated
            invalid_percent = 100.0
            message = (
                "Colunas ausentes: "
                + ", ".join(
                    missing_columns
                )
            )

        elif rule_type == "not_null":
            null_condition = None

            for column_name in resolved_columns:
                column_condition = (
                    F.col(column_name).isNull()
                    | (
                        F.trim(
                            F.col(column_name)
                            .cast("string")
                        ) == ""
                    )
                    | (
                        F.lower(
                            F.trim(
                                F.col(column_name)
                                .cast("string")
                            )
                        ) == "nan"
                    )
                )

                null_condition = (
                    column_condition
                    if null_condition is None
                    else (
                        null_condition
                        | column_condition
                    )
                )

            invalid_df = (
                df_dataset
                .filter(null_condition)
            )

            invalid_records = (
                invalid_df.count()
            )

            invalid_percent = (
                invalid_records
                / records_evaluated
                * 100
            )

            message = (
                f"{invalid_records} registros "
                f"com valor nulo nas colunas "
                f"{resolved_columns}."
            )

        elif rule_type == "unique":
            duplicate_keys = (
                df_dataset
                .groupBy(
                    *resolved_columns
                )
                .count()
                .filter(
                    F.col("count") > 1
                )
            )

            invalid_records = (
                duplicate_keys
                .agg(
                    F.sum(
                        F.col("count") - 1
                    ).alias("duplicates")
                )
                .first()["duplicates"]
                or 0
            )

            invalid_percent = (
                invalid_records
                / records_evaluated
                * 100
            )

            if invalid_records > 0:
                invalid_df = (
                    df_dataset
                    .join(
                        duplicate_keys
                        .drop("count"),
                        on=resolved_columns,
                        how="inner"
                    )
                )

            message = (
                f"{invalid_records} registros "
                f"duplicados para a chave "
                f"{resolved_columns}."
            )

        elif rule_type == "allowed_values":
            target_column = (
                resolved_columns[0]
            )

            if target_column == "IN_ALFABETIZADO":
                allowed_values = [
                    "0",
                    "1",
                    "0.0",
                    "1.0"
                ]
            else:
                allowed_values = [
                    "0",
                    "1"
                ]

            invalid_df = (
                df_dataset
                .filter(
                    F.col(target_column).isNotNull()
                    & ~(
                        F.trim(
                            F.col(target_column)
                            .cast("string")
                        )
                        .isin(allowed_values)
                    )
                )
            )

            invalid_records = (
                invalid_df.count()
            )

            invalid_percent = (
                invalid_records
                / records_evaluated
                * 100
            )

            message = (
                f"{invalid_records} registros "
                f"fora dos valores permitidos "
                f"{allowed_values}."
            )

        elif rule_type == "between":
            target_column = (
                resolved_columns[0]
            )

            numeric_value = numeric_column(
                target_column
            )

            invalid_df = (
                df_dataset
                .filter(
                    F.col(target_column).isNotNull()
                    & (
                        numeric_value.isNull()
                        | ~numeric_value.between(
                            0.0,
                            100.0
                        )
                    )
                )
            )

            invalid_records = (
                invalid_df.count()
            )

            invalid_percent = (
                invalid_records
                / records_evaluated
                * 100
            )

            message = (
                f"{invalid_records} registros "
                f"fora da faixa de 0 a 100 "
                f"na coluna {target_column}."
            )

        else:
            invalid_records = records_evaluated
            invalid_percent = 100.0
            message = (
                f"Tipo de regra não suportado: "
                f"{rule_type}."
            )

        status = calculate_status(
            invalid_percent,
            rule["tolerance_percent"]
        )

        if (
            invalid_df is not None
            and invalid_records > 0
        ):
            rejected_output_path = (
                f"{QUALITY_REJECTED_PATH}/"
                f"dataset={dataset}/"
                f"ano={ano}/"
                f"rule_id={rule['rule_id']}/"
                f"execution_date={EXECUTION_DATE}"
            )

            (
                invalid_df
                .withColumn(
                    "_quality_rule_id",
                    F.lit(rule["rule_id"])
                )
                .withColumn(
                    "_quality_reason",
                    F.lit(message)
                )
                .withColumn(
                    "_quality_execution_date",
                    F.lit(EXECUTION_DATE)
                )
                .coalesce(1)
                .write
                .mode("overwrite")
                .format("parquet")
                .option(
                    "compression",
                    "snappy"
                )
                .save(
                    rejected_output_path
                )
            )

            rejected_records_paths.append({
                "dataset": dataset,
                "ano": ano,
                "rule_id": rule["rule_id"],
                "path": rejected_output_path,
                "invalid_records": int(
                    invalid_records
                )
            })

        quality_results.append({
            "layer": "silver",
            "dataset": dataset,
            "ano": ano,
            "rule_id": rule["rule_id"],
            "rule_name": rule["rule_name"],
            "rule_type": rule_type,
            "column_name": rule["column_name"],
            "severity": rule["severity"],
            "blocking": bool(
                rule["blocking"]
            ),
            "tolerance_percent": float(
                rule["tolerance_percent"]
            ),
            "status": status,
            "records_evaluated": int(
                records_evaluated
            ),
            "invalid_records": int(
                invalid_records
            ),
            "invalid_percent": float(
                round(
                    invalid_percent,
                    4
                )
            ),
            "message": message,
            "evaluated_path": silver_file_path,
            "execution_date": str(
                EXECUTION_DATE
            ),
            "evaluated_at": (
                datetime.now()
                .isoformat()
            )
        })

print(
    "Validações executadas:",
    len(quality_results)
)

print(
    "Conjuntos de rejeitados:",
    len(rejected_records_paths)
)

## 11. Criação do DataFrame de resultados

O schema explícito evita falhas quando nenhuma regra encontra registros inválidos.

In [0]:
df_quality_results = spark.createDataFrame(
    quality_results,
    schema=schema_quality_result
)

display(
    df_quality_results
    .orderBy(
        "dataset",
        "ano",
        "rule_id"
    )
)

## 12. Persistência dos resultados detalhados

Os resultados detalhados são armazenados por data de execução.

In [0]:
details_output_path = (
    f"{QUALITY_DETAILS_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_results
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option(
        "compression",
        "snappy"
    )
    .save(details_output_path)
)

print("Detalhes salvos em:")
print(details_output_path)

## 13. Criação do resumo por dataset

O resumo apresenta:

- total de regras;
- regras aprovadas;
- regras em atenção;
- regras reprovadas;
- regras bloqueantes reprovadas;
- score de qualidade.

In [0]:
df_quality_summary = (
    df_quality_results
    .groupBy(
        "layer",
        "dataset",
        "ano"
    )
    .agg(
        F.count("*").alias(
            "total_rules"
        ),
        F.sum(
            F.when(
                F.col("status")
                == "APROVADO",
                1
            ).otherwise(0)
        ).alias(
            "approved_rules"
        ),
        F.sum(
            F.when(
                F.col("status")
                == "ATENCAO",
                1
            ).otherwise(0)
        ).alias(
            "warning_rules"
        ),
        F.sum(
            F.when(
                F.col("status")
                == "REPROVADO",
                1
            ).otherwise(0)
        ).alias(
            "failed_rules"
        ),
        F.sum(
            F.when(
                (
                    F.col("status")
                    == "REPROVADO"
                )
                & F.col("blocking"),
                1
            ).otherwise(0)
        ).alias(
            "blocking_failed_rules"
        ),
        F.sum(
            F.col("invalid_records")
        ).alias(
            "total_invalid_records"
        )
    )
    .withColumn(
        "quality_score_percent",
        F.round(
            (
                F.col("approved_rules")
                / F.col("total_rules")
            ) * 100,
            2
        )
    )
    .withColumn(
        "dataset_status",
        F.when(
            F.col(
                "blocking_failed_rules"
            ) > 0,
            "REPROVADO"
        )
        .when(
            F.col(
                "failed_rules"
            ) > 0,
            "ATENCAO"
        )
        .otherwise(
            "APROVADO"
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_quality_summary
    .orderBy(
        "dataset",
        "ano"
    )
)

## 14. Persistência do resumo Silver

O resumo será utilizado pelo dashboard de qualidade e pelo monitoramento.

In [0]:
summary_output_path = (
    f"{QUALITY_SUMMARY_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option(
        "compression",
        "snappy"
    )
    .save(summary_output_path)
)

print("Resumo salvo em:")
print(summary_output_path)

## 15. Histórico de qualidade Silver

Os resultados detalhados são adicionados ao histórico para análise temporal.

In [0]:
(
    df_quality_results
    .write
    .mode("append")
    .format("parquet")
    .partitionBy(
        "execution_date"
    )
    .save(
        QUALITY_HISTORY_PATH
    )
)

print("Histórico atualizado em:")
print(QUALITY_HISTORY_PATH)

## 16. Resumo dos registros rejeitados

Esta visão permite identificar quais regras produziram registros inválidos e onde eles foram persistidos.

In [0]:
schema_rejected_summary = StructType([
    StructField(
        "dataset",
        StringType(),
        False
    ),
    StructField(
        "ano",
        IntegerType(),
        False
    ),
    StructField(
        "rule_id",
        StringType(),
        False
    ),
    StructField(
        "path",
        StringType(),
        False
    ),
    StructField(
        "invalid_records",
        LongType(),
        False
    )
])

df_rejected_summary = (
    spark.createDataFrame(
        rejected_records_paths,
        schema=schema_rejected_summary
    )
)

if df_rejected_summary.count() == 0:
    print(
        "Nenhum conjunto de registros "
        "rejeitados foi gerado."
    )
else:
    display(
        df_rejected_summary
        .orderBy(
            "dataset",
            "ano",
            "rule_id"
        )
    )

## 17. Diagnóstico das regras bloqueantes

Antes do checklist final, esta etapa apresenta as regras críticas reprovadas, caso existam.

O diagnóstico não altera a classificação nem flexibiliza a regra. Ele apenas fornece evidências para identificar:

- dataset;
- ano;
- chave avaliada;
- quantidade de registros inválidos;
- caminho dos rejeitados;
- causa da reprovação.

In [0]:
df_bloqueios = (
    df_quality_results
    .filter(
        (F.col("status") == "REPROVADO")
        & F.col("blocking")
    )
    .select(
        "dataset",
        "ano",
        "rule_id",
        "rule_name",
        "column_name",
        "severity",
        "records_evaluated",
        "invalid_records",
        "invalid_percent",
        "message",
        "evaluated_path"
    )
    .orderBy(
        "dataset",
        "ano",
        "rule_id"
    )
)

quantidade_bloqueios = (
    df_bloqueios.count()
)

if quantidade_bloqueios == 0:

    print(
        "Nenhuma regra bloqueante "
        "foi reprovada."
    )

else:

    print(
        "Regras bloqueantes reprovadas:",
        quantidade_bloqueios
    )

    display(df_bloqueios)

## 17. Checklist final

O notebook verifica se existem regras bloqueantes reprovadas.

Os resultados são persistidos antes da interrupção para garantir auditoria.

### Comportamento definitivo

As regras críticas permanecem bloqueantes e com tolerância zero.

A execução somente será concluída quando:

```text
regras bloqueantes reprovadas = 0
```

Caso exista reprovação, o notebook mantém os resultados persistidos para auditoria e interrompe o pipeline antes da camada Gold.

In [0]:
blocking_failures = (
    df_quality_results
    .filter(
        (F.col("status") == "REPROVADO")
        & F.col("blocking")
    )
)

blocking_failure_count = (
    blocking_failures.count()
)

if blocking_failure_count > 0:

    display(
        blocking_failures
        .orderBy(
            "dataset",
            "ano",
            "rule_id"
        )
    )

    raise Exception(
        f"Foram encontradas "
        f"{blocking_failure_count} "
        f"regras bloqueantes reprovadas "
        f"na camada Silver. "
        f"Consulte o diagnóstico e os "
        f"registros rejeitados antes de "
        f"prosseguir para a camada Gold."
    )

print(
    "Data Quality Silver concluída "
    "com sucesso."
)

print(
    "Todas as regras críticas "
    "foram aprovadas."
)

## Resultado esperado

Ao final deste notebook estarão disponíveis:

```text
logs/data_quality/silver/details/execution_date=YYYY-MM-DD
logs/data_quality/silver/summary/execution_date=YYYY-MM-DD
logs/data_quality/rejected/silver/
logs/data_quality/history/silver
```

### Próximo notebook

```text
05_3_quality_gold
```